# Experiment Outputs Dashboard

This notebook discovers experiment outputs and plots each artifact type in separate sections for clean comparison across runs.

In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [ ]:
KNOWN_ARTIFACTS = {
    "assignments.parquet",
    "per_document_utility.parquet",
    "pt_experiment.csv",
    "summary.json",
    "memory_summary.json",
    "full_run.parquet",
    "optimized_run.parquet",
    "profile_catalog.csv",
    "sampled_score_pairs.parquet",
    "metrics_long.csv",
    "metrics_wide.csv",
    "ranking_summary.csv",
    "run_metadata.json",
}

BASE_OUTPUT_DIR = (Path.cwd().parent / "matryoshka_optimization_codebase" / "outputs").resolve()
EXPORT_ROOT = (Path.cwd() / "exports").resolve()

print("Base outputs dir:", BASE_OUTPUT_DIR)
print("Export root:", EXPORT_ROOT)

In [ ]:
def _is_excluded_path(path: Path) -> bool:
    parts = {p.lower() for p in path.parts}
    return "models" in parts


def _safe_read(path: Path):
    suffix = path.suffix.lower()
    try:
        if suffix == ".parquet":
            return pd.read_parquet(path), None
        if suffix == ".csv":
            return pd.read_csv(path), None
        if suffix == ".json":
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f), None
        return None, f"Unsupported extension: {suffix}"
    except Exception as exc:
        return None, str(exc)


def infer_experiment_type(artifact_names: set[str]) -> str:
    if "assignments.parquet" in artifact_names or "per_document_utility.parquet" in artifact_names:
        return "matryoshka_run"
    if {"metrics_long.csv", "metrics_wide.csv"}.intersection(artifact_names):
        return "side_quest_benchmark"
    if "pt_experiment.csv" in artifact_names and "summary.json" not in artifact_names:
        return "side_quest_benchmark"
    return "generic_experiment"


def discover_runs(base_output_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    if not base_output_dir.exists():
        return pd.DataFrame(columns=["run_name", "run_path", "experiment_type", "artifact_count", "artifacts"])

    for run_dir in sorted(p for p in base_output_dir.rglob("*") if p.is_dir()):
        if _is_excluded_path(run_dir):
            continue

        artifact_paths = [p for p in run_dir.iterdir() if p.is_file() and p.name in KNOWN_ARTIFACTS]
        if not artifact_paths:
            continue

        artifact_names = {p.name for p in artifact_paths}
        rows.append(
            {
                "run_name": run_dir.name,
                "run_path": str(run_dir),
                "experiment_type": infer_experiment_type(artifact_names),
                "artifact_count": len(artifact_names),
                "artifacts": sorted(artifact_names),
            }
        )

    return pd.DataFrame(rows).sort_values(["experiment_type", "run_name"]).reset_index(drop=True) if rows else pd.DataFrame(
        columns=["run_name", "run_path", "experiment_type", "artifact_count", "artifacts"]
    )


def load_run_artifacts(run_path: str | Path) -> dict[str, Any]:
    run_path = Path(run_path)
    bundle: dict[str, Any] = {
        "run_name": run_path.name,
        "run_path": run_path,
        "tables": {},
        "json": {},
        "errors": {},
        "schema": {},
    }

    for artifact in KNOWN_ARTIFACTS:
        artifact_path = run_path / artifact
        if not artifact_path.exists():
            continue
        data, err = _safe_read(artifact_path)
        if err:
            bundle["errors"][artifact] = err
            continue

        if isinstance(data, pd.DataFrame):
            bundle["tables"][artifact] = data
            bundle["schema"][artifact] = {
                "rows": int(len(data)),
                "columns": list(data.columns),
            }
        elif isinstance(data, dict):
            bundle["json"][artifact] = data
            bundle["schema"][artifact] = {
                "keys": sorted(list(data.keys())),
            }

    return bundle


def build_manifest_with_schema(manifest: pd.DataFrame) -> pd.DataFrame:
    if manifest.empty:
        return manifest

    rows = []
    for _, row in manifest.iterrows():
        bundle = load_run_artifacts(row["run_path"])
        table_row_counts = {
            name: meta.get("rows")
            for name, meta in bundle["schema"].items()
            if "rows" in meta
        }
        key_columns = {
            name: meta.get("columns", [])[:8]
            for name, meta in bundle["schema"].items()
            if "columns" in meta
        }
        rows.append(
            {
                **row.to_dict(),
                "table_row_counts": table_row_counts,
                "key_columns": key_columns,
                "load_errors": bundle["errors"],
            }
        )
    return pd.DataFrame(rows)


def _first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    return None


NON_METRIC_COLUMNS = {
    "name",
    "run",
    "system",
    "model",
    "model_id",
    "model_type",
    "dimension",
    "profile",
    "profile_name",
    "docno",
    "qid",
    "rank",
    "cost_bytes",
    "utility",
}

STATISTICAL_COLUMN_MARKERS = (
    "p-value",
    "p_value",
    "pvalue",
    "pval",
    "significant",
    "significance",
    "reject",
    "better",
    "worse",
    "improved",
    "degraded",
    "stderr",
    "std_err",
    "std error",
    "stddev",
    "variance",
    "ci",
    "confidence",
    "test",
    "correction",
)


def _is_statistical_or_nonmetric_column(column: str) -> bool:
    name = str(column).strip().lower()
    compact = name.replace(" ", "_")
    if compact in NON_METRIC_COLUMNS:
        return True
    if name.endswith(" +") or name.endswith(" -"):
        return True
    if any(marker in name for marker in STATISTICAL_COLUMN_MARKERS):
        return True
    return False


def _evaluation_metric_cols(df: pd.DataFrame) -> list[str]:
    return [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c]) and not _is_statistical_or_nonmetric_column(str(c))
    ]


def _format_count(value: int | float) -> str:
    return f"{int(value):,}".replace(",", " ")


def _short_run_label(name: str) -> str:
    label = str(name)
    label = label.replace("profile_", "")
    label = label.replace("_run", "")
    label = label.replace("_embedding", "")
    return label

## Discover Experiment Runs

This cell discovers run folders under `outputs/`, excludes `models` paths, and builds the run manifest.

In [ ]:
manifest = discover_runs(BASE_OUTPUT_DIR)
if manifest.empty:
    print("No experiment outputs discovered yet. Add run folders under:")
    print(BASE_OUTPUT_DIR)
else:
    display(manifest)

## Schema Profiling

This cell loads discovered artifacts safely and summarizes row counts, key columns, and loading errors per run.

In [ ]:
manifest_profiled = build_manifest_with_schema(manifest)
if not manifest_profiled.empty:
    display(manifest_profiled[["run_name", "experiment_type", "artifact_count", "table_row_counts", "key_columns", "load_errors"]])

## Load All Run Bundles

This cell prepares a bundle per run so each subsequent section can compare one artifact type across all runs.

In [ ]:
run_bundles = []
if manifest.empty:
    print("No run bundles loaded because discovery found no runs.")
else:
    for _, row in manifest.iterrows():
        run_bundles.append(load_run_artifacts(row["run_path"]))
    print(f"Loaded {len(run_bundles)} run bundles.")

## Assignments Comparison (`assignments.parquet`)

Plots assigned documents per profile with horizontal log-scaled bars. Exact document counts and percentages are written next to each bar so small profiles remain visible even when one profile dominates.

In [ ]:
if not run_bundles:
    print("No bundles available.")
else:
    plotted = 0
    for b in run_bundles:
        df = b["tables"].get("assignments.parquet")
        if df is None or df.empty:
            continue

        profile_col = _first_existing_col(df, ["profile", "profile_name", "assigned_profile", "name"])
        cost_col = _first_existing_col(df, ["cost_bytes", "cost"])
        if not profile_col:
            continue

        counts = df[profile_col].astype(str).value_counts()
        catalog = b["tables"].get("profile_catalog.csv")
        if isinstance(catalog, pd.DataFrame) and "name" in catalog.columns:
            profile_order = [str(x) for x in catalog["name"].tolist()]
            counts = counts.reindex(profile_order, fill_value=0)
        else:
            counts = counts.sort_values(ascending=True)

        total_docs = int(counts.sum())
        plot_counts = counts.clip(lower=1)
        fig_height = max(4, 0.55 * len(counts) + 1.5)
        fig, ax = plt.subplots(figsize=(10, fig_height))
        bars = ax.barh(counts.index.astype(str), plot_counts.values)
        ax.set_xscale("log")
        ax.set_title(f"{b['run_name']} - Assigned Documents per Profile (log scale)")
        ax.set_xlabel("Documents assigned (log scale; labels show exact values)")
        ax.set_ylabel("Profile")
        ax.grid(axis="x", which="both", alpha=0.3)

        x_max = max(float(plot_counts.max()), 1.0)
        ax.set_xlim(0.8, x_max * 8.0)
        for bar, profile_name, count in zip(bars, counts.index.astype(str), counts.values):
            pct = (100.0 * count / total_docs) if total_docs else 0.0
            label = f"{_format_count(count)} docs ({pct:.2f}%)"
            ax.text(
                max(float(count), 1.0) * 1.12,
                bar.get_y() + bar.get_height() / 2,
                label,
                va="center",
                ha="left",
                fontsize=9,
            )

        plt.tight_layout()
        plt.show()
        plotted += 1

        if cost_col and pd.api.types.is_numeric_dtype(df[cost_col]):
            cost_summary = (
                df.groupby(profile_col)[cost_col]
                .agg(documents="count", total_cost_bytes="sum", avg_cost_bytes="mean")
                .reset_index()
                .sort_values("documents", ascending=False)
            )
            display(cost_summary)

    if plotted == 0:
        print("No plottable assignments tables found.")

## Utility Comparison (`per_document_utility.parquet`)

Fast comparison plot: one line per profile, with `x` as document index and `y` as utility.

In [ ]:
if not run_bundles:
    print("No bundles available.")
else:
    plotted = 0
    max_points_per_profile = 10000

    for b in run_bundles:
        df = b["tables"].get("per_document_utility.parquet")
        if df is None or df.empty:
            continue

        utility_col = _first_existing_col(df, ["utility", "mean_utility", "u", "score"])
        if utility_col is None:
            numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
            utility_col = numeric_cols[0] if numeric_cols else None
        if utility_col is None:
            continue

        profile_col = _first_existing_col(df, ["profile", "profile_name", "candidate_profile"])
        doc_col = _first_existing_col(df, ["docno", "doc_id", "document_id"])

        if profile_col is None:
            # Single-line fallback when no profile column is available.
            tmp = df[[utility_col]].copy()
            tmp = tmp.dropna(subset=[utility_col]).reset_index(drop=True)
            if tmp.empty:
                continue
            if len(tmp) > max_points_per_profile:
                step = max(1, len(tmp) // max_points_per_profile)
                tmp = tmp.iloc[::step].reset_index(drop=True)

            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(tmp.index.values, tmp[utility_col].values, linewidth=1.0, label="utility")
            ax.set_title(f"{b['run_name']} - Utility Line Plot")
            ax.set_xlabel("Document index")
            ax.set_ylabel(utility_col)
            ax.legend()
            plt.tight_layout()
            plt.show()
            plotted += 1
            continue

        cols = [profile_col, utility_col]
        if doc_col:
            cols.append(doc_col)
        tmp = df[cols].copy().dropna(subset=[utility_col])
        if tmp.empty:
            continue

        # Stable ordering: by document id when present, else by original row order.
        if doc_col:
            tmp[doc_col] = tmp[doc_col].astype(str)
            tmp = tmp.sort_values([profile_col, doc_col]).reset_index(drop=True)
        else:
            tmp = tmp.reset_index(drop=True)

        fig, ax = plt.subplots(figsize=(11, 5))
        for profile_name, grp in tmp.groupby(profile_col, sort=True):
            grp = grp.reset_index(drop=True)
            if len(grp) > max_points_per_profile:
                step = max(1, len(grp) // max_points_per_profile)
                grp = grp.iloc[::step].reset_index(drop=True)
            ax.plot(grp.index.values, grp[utility_col].values, linewidth=1.0, label=str(profile_name))

        ax.set_title(f"{b['run_name']} - Utility by Document (One Line per Profile)")
        ax.set_xlabel("Document index")
        ax.set_ylabel(utility_col)
        ax.legend(title="Profile", loc="best", fontsize=8)
        plt.tight_layout()
        plt.show()
        plotted += 1

    if plotted == 0:
        print("No plottable per_document_utility tables found.")

## Evaluation Comparison (`pt_experiment.csv`, `metrics_wide.csv`)

Plots only retrieval metric columns from evaluation artifacts. Statistical-test columns such as p-values, significance flags, confidence intervals, and plus/minus columns are filtered out.

In [ ]:
if not run_bundles:
    print("No bundles available.")
else:
    plotted = 0
    for b in run_bundles:
        df = b["tables"].get("pt_experiment.csv")
        run_col = None

        if df is not None and not df.empty:
            run_col = _first_existing_col(df, ["name", "system", "run", "model"])
        else:
            wide_df = b["tables"].get("metrics_wide.csv")
            if wide_df is not None and not wide_df.empty:
                df = wide_df.copy()
                if "model_id" in df.columns and "dimension" in df.columns:
                    df["__run_label"] = df["model_id"].astype(str) + "_d" + df["dimension"].astype(str)
                    run_col = "__run_label"
                else:
                    run_col = _first_existing_col(df, ["model_id", "name", "system", "run", "model"])
            else:
                continue

        metric_cols = _evaluation_metric_cols(df)
        if not run_col or not metric_cols:
            print(f"{b['run_name']}: no metric columns found after filtering statistical-test columns.")
            continue

        plot_df = df[[run_col] + metric_cols].copy()
        plot_df[run_col] = plot_df[run_col].astype(str).map(_short_run_label)
        melted = plot_df.melt(id_vars=[run_col], value_vars=metric_cols, var_name="metric", value_name="value")

        fig_width = max(10, 1.4 * len(metric_cols))
        fig, ax = plt.subplots(figsize=(fig_width, 5))
        if sns is not None:
            sns.barplot(data=melted, x="metric", y="value", hue=run_col, ax=ax)
        else:
            for label, grp in melted.groupby(run_col):
                ax.plot(grp["metric"], grp["value"], marker="o", label=label)
            ax.legend()
        ax.set_title(f"{b['run_name']} - Evaluation Metrics")
        ax.set_xlabel("Metric")
        ax.set_ylabel("Value")
        plt.xticks(rotation=30, ha="right")
        ax.legend(title="Run", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
        plotted += 1

        baseline = df.iloc[0]
        baseline_name = _short_run_label(str(baseline[run_col]))
        delta_rows = []
        for _, row in df.iterrows():
            current_name = _short_run_label(str(row[run_col]))
            for m in metric_cols:
                delta_rows.append(
                    {
                        "run_name": current_name,
                        "metric": m,
                        "delta_vs_first_run": float(row[m] - baseline[m]),
                    }
                )

        delta_df = pd.DataFrame(delta_rows)
        if not delta_df.empty:
            fig, ax = plt.subplots(figsize=(fig_width, 5))
            if sns is not None:
                sns.barplot(data=delta_df, x="metric", y="delta_vs_first_run", hue="run_name", ax=ax)
            else:
                for name, grp in delta_df.groupby("run_name"):
                    ax.plot(grp["metric"], grp["delta_vs_first_run"], marker="o", label=name)
                ax.legend()
            ax.axhline(0.0, color="black", linewidth=1, linestyle="--")
            ax.set_title(f"{b['run_name']} - Metric Delta vs First Run ({baseline_name})")
            ax.set_xlabel("Metric")
            ax.set_ylabel("Delta")
            plt.xticks(rotation=30, ha="right")
            ax.legend(title="Run", bbox_to_anchor=(1.02, 1), loc="upper left")
            plt.tight_layout()
            plt.show()
            plotted += 1

    if plotted == 0:
        print("No plottable evaluation tables found (`pt_experiment.csv` or `metrics_wide.csv`).")

## Summary and Memory Comparison (`summary.json`, `memory_summary.json`)

Displays compact KPI tables and memory usage bars per run using summary JSON artifacts.

In [ ]:
if not run_bundles:
    print("No bundles available.")
else:
    plotted = 0
    kpi_rows = []
    for b in run_bundles:
        summary = b["json"].get("summary.json", {})
        memory = b["json"].get("memory_summary.json", {})
        if not summary and not memory:
            continue

        row = {
            "run_name": b["run_name"],
            "lambda_star": summary.get("lambda_star"),
            "optimized_total_utility": summary.get("optimized_total_utility"),
            "num_docs": summary.get("num_docs"),
            "num_queries": summary.get("num_queries"),
            "budget_bytes": memory.get("budget_bytes"),
            "optimized_total_cost_bytes": memory.get("optimized_total_cost_bytes"),
            "full_total_cost_bytes": memory.get("full_total_cost_bytes"),
        }
        kpi_rows.append(row)

        if memory.get("optimized_total_cost_bytes") is not None and memory.get("full_total_cost_bytes") is not None:
            s = pd.Series({
                "optimized_total_cost_bytes": memory["optimized_total_cost_bytes"],
                "full_total_cost_bytes": memory["full_total_cost_bytes"],
            })
            fig, ax = plt.subplots(figsize=(8, 4))
            s.plot(kind="bar", ax=ax)
            ax.set_title(f"{b['run_name']} - Memory Usage")
            ax.set_xlabel("Mode")
            ax.set_ylabel("Bytes")
            plt.tight_layout()
            plt.show()
            plotted += 1

    if kpi_rows:
        display(pd.DataFrame(kpi_rows).sort_values("run_name").reset_index(drop=True))
    if plotted == 0 and not kpi_rows:
        print("No summary/memory JSON data found.")

## Retrieval Runs Comparison (`full_run.parquet`, `optimized_run.parquet`)

Compares score distributions and per-query top-k overlap between full and optimized runs for each experiment.

In [ ]:
if not run_bundles:
    print("No bundles available.")
else:
    plotted = 0
    for b in run_bundles:
        full_df = b["tables"].get("full_run.parquet")
        opt_df = b["tables"].get("optimized_run.parquet")
        if full_df is None or opt_df is None or full_df.empty or opt_df.empty:
            continue

        full_score_col = _first_existing_col(full_df, ["score", "sim", "similarity"])
        opt_score_col = _first_existing_col(opt_df, ["score", "sim", "similarity"])

        if full_score_col and opt_score_col:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(full_df[full_score_col].dropna(), bins=40, alpha=0.6, label="full")
            ax.hist(opt_df[opt_score_col].dropna(), bins=40, alpha=0.6, label="optimized")
            ax.set_title(f"{b['run_name']} - Retrieval Score Distribution")
            ax.set_xlabel("Score")
            ax.set_ylabel("Count")
            ax.legend()
            plt.tight_layout()
            plt.show()
            plotted += 1

        qid_col_full = _first_existing_col(full_df, ["qid", "query_id"])
        doc_col_full = _first_existing_col(full_df, ["docno", "doc_id"])
        qid_col_opt = _first_existing_col(opt_df, ["qid", "query_id"])
        doc_col_opt = _first_existing_col(opt_df, ["docno", "doc_id"])

        if qid_col_full and doc_col_full and qid_col_opt and doc_col_opt:
            overlaps = []
            full_groups = full_df.groupby(qid_col_full)[doc_col_full].apply(set)
            opt_groups = opt_df.groupby(qid_col_opt)[doc_col_opt].apply(set)
            common_qids = sorted(set(full_groups.index).intersection(set(opt_groups.index)))
            for qid in common_qids:
                a = full_groups.loc[qid]
                bset = opt_groups.loc[qid]
                denom = max(len(a), 1)
                overlaps.append(len(a.intersection(bset)) / denom)

            if overlaps:
                fig, ax = plt.subplots(figsize=(8, 4))
                ax.hist(overlaps, bins=25)
                ax.set_title(f"{b['run_name']} - Top-k Overlap Ratio per Query")
                ax.set_xlabel("Overlap ratio")
                ax.set_ylabel("Queries")
                plt.tight_layout()
                plt.show()
                plotted += 1

    if plotted == 0:
        print("No full/optimized run pairs found for plotting.")

## Cross-Run Summary Builder

Builds a single comparison table with core KPIs, memory savings, and evaluation metrics across all runs.

In [ ]:
def build_cross_run_summary(run_bundles: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for b in run_bundles:
        run_name = b["run_name"]
        summary = b.get("json", {}).get("summary.json", {})
        memory = b.get("json", {}).get("memory_summary.json", {})
        pt_eval = b.get("tables", {}).get("pt_experiment.csv")
        wide_eval = b.get("tables", {}).get("metrics_wide.csv")

        base_row = {
            "experiment_run_name": run_name,
            "lambda_star": summary.get("lambda_star"),
            "optimized_total_utility": summary.get("optimized_total_utility"),
            "num_docs": summary.get("num_docs"),
            "num_queries": summary.get("num_queries"),
            "num_eval_queries": summary.get("num_eval_queries"),
            "budget_bytes": memory.get("budget_bytes"),
            "optimized_total_cost_bytes": memory.get("optimized_total_cost_bytes"),
            "full_total_cost_bytes": memory.get("full_total_cost_bytes"),
        }
        if base_row["optimized_total_cost_bytes"] is not None and base_row["full_total_cost_bytes"] is not None and base_row["full_total_cost_bytes"]:
            base_row["memory_saved_pct"] = 100.0 * (1.0 - (base_row["optimized_total_cost_bytes"] / base_row["full_total_cost_bytes"]))
        else:
            base_row["memory_saved_pct"] = np.nan

        eval_df = None
        run_col = None
        if isinstance(pt_eval, pd.DataFrame) and not pt_eval.empty:
            eval_df = pt_eval.copy()
            run_col = _first_existing_col(eval_df, ["name", "run", "system", "model"])
        elif isinstance(wide_eval, pd.DataFrame) and not wide_eval.empty:
            eval_df = wide_eval.copy()
            if "model_id" in eval_df.columns and "dimension" in eval_df.columns:
                eval_df["__run_label"] = eval_df["model_id"].astype(str) + "_d" + eval_df["dimension"].astype(str)
                run_col = "__run_label"
            else:
                run_col = _first_existing_col(eval_df, ["model_id", "name", "run", "system", "model"])

        if isinstance(eval_df, pd.DataFrame) and not eval_df.empty:
            metric_cols = _evaluation_metric_cols(eval_df)

            if run_col and metric_cols:
                baseline = eval_df.iloc[0]
                baseline_name = _short_run_label(str(baseline[run_col]))

                for _, row in eval_df.iterrows():
                    out = dict(base_row)
                    current_name = _short_run_label(str(row[run_col]))
                    out["eval_run_name"] = current_name
                    out["eval_baseline_name"] = baseline_name
                    out["is_baseline"] = current_name == baseline_name

                    for m in metric_cols:
                        out[f"metric__{m}"] = float(row[m])
                        out[f"metric_delta_vs_baseline__{m}"] = float(row[m] - baseline[m])

                    rows.append(out)
            else:
                rows.append(base_row)
        else:
            rows.append(base_row)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    sort_cols = [c for c in ["experiment_run_name", "eval_run_name"] if c in df.columns]
    return df.sort_values(sort_cols, na_position="last").reset_index(drop=True) if sort_cols else df


cross_summary = build_cross_run_summary(run_bundles)
if cross_summary.empty:
    print("No cross-run summary available yet.")
else:
    display(cross_summary)

## Cross-Run Metric Comparison Plot

Plots grouped bars of optimized evaluation metrics across experiments.

In [ ]:
if cross_summary.empty:
    print("No cross-run summary available.")
else:
    metric_cols = [c for c in cross_summary.columns if c.startswith("metric__") and not c.startswith("metric_delta_vs_baseline__")]
    if not metric_cols:
        print("No metric columns found.")
    else:
        id_col = "eval_run_name" if "eval_run_name" in cross_summary.columns else "experiment_run_name"
        melted = cross_summary.melt(id_vars=["experiment_run_name", id_col], value_vars=metric_cols, var_name="metric", value_name="value")
        melted["metric"] = melted["metric"].str.replace("metric__", "", regex=False)
        melted["label"] = melted["experiment_run_name"] + " :: " + melted[id_col].fillna("n/a")

        fig, ax = plt.subplots(figsize=(12, 5))
        if sns is not None:
            sns.barplot(data=melted, x="metric", y="value", hue="label", ax=ax)
        else:
            for label, grp in melted.groupby("label"):
                ax.plot(grp["metric"], grp["value"], marker="o", label=label)
            ax.legend()
        ax.set_title("Cross-Run Metrics (All Evaluation Runs)")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

## Cross-Run Tradeoff Scatter

Plots memory savings versus the first available metric delta to visualize quality-memory tradeoffs.

In [ ]:
if cross_summary.empty:
    print("No cross-run summary available.")
else:
    delta_cols = [c for c in cross_summary.columns if c.startswith("metric_delta_vs_baseline__")]
    if not delta_cols:
        print("No baseline-delta metric columns found.")
    else:
        primary_delta = delta_cols[0]
        plot_df = cross_summary.dropna(subset=["memory_saved_pct", primary_delta]).copy()
        if plot_df.empty:
            print("Not enough data for tradeoff scatter.")
        else:
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.scatter(plot_df["memory_saved_pct"], plot_df[primary_delta])
            labels = []
            for _, r in plot_df.iterrows():
                exp_name = r.get("experiment_run_name", "")
                eval_name = r.get("eval_run_name", "")
                label = f"{exp_name}::{eval_name}" if eval_name else exp_name
                labels.append(label)
                ax.annotate(label, (r["memory_saved_pct"], r[primary_delta]), fontsize=8)
            ax.set_title(f"Memory Saved vs {primary_delta.replace('metric_delta_vs_baseline__', '')} Delta")
            ax.set_xlabel("Memory saved (%)")
            ax.set_ylabel("Metric delta vs baseline")
            plt.tight_layout()
            plt.show()

## Cross-Run Ranking Table

Shows runs ranked by the first available optimized metric (or delta metric when optimized metrics are unavailable).

In [ ]:
if cross_summary.empty:
    print("No cross-run summary available.")
else:
    metric_cols = [c for c in cross_summary.columns if c.startswith("metric__") and not c.startswith("metric_delta_vs_baseline__")]
    delta_cols = [c for c in cross_summary.columns if c.startswith("metric_delta_vs_baseline__")]

    ranking_col = metric_cols[0] if metric_cols else (delta_cols[0] if delta_cols else None)

    if ranking_col is None:
        print("No ranking metric column found.")
    else:
        cols = ["experiment_run_name", "eval_run_name", ranking_col, "memory_saved_pct", "optimized_total_utility"]
        cols = [c for c in cols if c in cross_summary.columns]
        ranking = cross_summary.sort_values(ranking_col, ascending=False)[cols]
        display(ranking)

## Export Dashboard Outputs

Exports manifest, cross-run summary, and artifact-specific plots to `analysis_viz/exports/<timestamp>/`.

In [ ]:
def export_dashboard_outputs(
    manifest_df: pd.DataFrame,
    cross_summary_df: pd.DataFrame,
    run_bundles_in: list[dict[str, Any]],
    export_root: Path,
) -> Path:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = export_root / ts
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest_df.to_csv(out_dir / "run_manifest.csv", index=False)
    cross_summary_df.to_csv(out_dir / "cross_run_summary.csv", index=False)

    for bundle in run_bundles_in:
        run_dir = out_dir / bundle["run_name"]
        run_dir.mkdir(parents=True, exist_ok=True)

        assignments = bundle["tables"].get("assignments.parquet")
        if assignments is not None and not assignments.empty:
            profile_col = _first_existing_col(assignments, ["profile", "profile_name", "assigned_profile", "name"])
            if profile_col:
                counts = assignments[profile_col].astype(str).value_counts().sort_values(ascending=True)
                plot_counts = counts.clip(lower=1)
                fig, ax = plt.subplots(figsize=(10, max(4, 0.55 * len(counts) + 1.5)))
                bars = ax.barh(counts.index.astype(str), plot_counts.values)
                ax.set_xscale("log")
                ax.set_title(f"{bundle['run_name']} - Assigned Documents per Profile")
                ax.set_xlabel("Documents assigned (log scale)")
                total_docs = int(counts.sum())
                ax.set_xlim(0.8, max(float(plot_counts.max()), 1.0) * 8.0)
                for bar, count in zip(bars, counts.values):
                    pct = (100.0 * count / total_docs) if total_docs else 0.0
                    ax.text(max(float(count), 1.0) * 1.12, bar.get_y() + bar.get_height() / 2, f"{_format_count(count)} ({pct:.2f}%)", va="center", fontsize=9)
                plt.tight_layout()
                fig.savefig(run_dir / "assignments_profile_counts.png", dpi=150, bbox_inches="tight")
                plt.close(fig)

        utility = bundle["tables"].get("per_document_utility.parquet")
        if utility is not None and not utility.empty:
            utility_col = _first_existing_col(utility, ["utility", "mean_utility", "u", "score"])
            if utility_col is None:
                num_cols = [c for c in utility.columns if pd.api.types.is_numeric_dtype(utility[c])]
                utility_col = num_cols[0] if num_cols else None
            if utility_col:
                profile_col = _first_existing_col(utility, ["profile", "profile_name", "candidate_profile"])
                doc_col = _first_existing_col(utility, ["docno", "doc_id", "document_id"])
                cols = [utility_col] + ([profile_col] if profile_col else []) + ([doc_col] if doc_col else [])
                tmp = utility[cols].copy().dropna(subset=[utility_col]).reset_index(drop=True)
                if not tmp.empty:
                    fig, ax = plt.subplots(figsize=(11, 5))
                    if profile_col:
                        if doc_col:
                            tmp[doc_col] = tmp[doc_col].astype(str)
                            tmp = tmp.sort_values([profile_col, doc_col]).reset_index(drop=True)
                        for profile_name, grp in tmp.groupby(profile_col, sort=True):
                            grp = grp.reset_index(drop=True)
                            if len(grp) > 10000:
                                step = max(1, len(grp) // 10000)
                                grp = grp.iloc[::step].reset_index(drop=True)
                            ax.plot(grp.index.values, grp[utility_col].values, linewidth=1.0, label=str(profile_name))
                        ax.legend(title="Profile", loc="best", fontsize=8)
                    else:
                        if len(tmp) > 10000:
                            step = max(1, len(tmp) // 10000)
                            tmp = tmp.iloc[::step].reset_index(drop=True)
                        ax.plot(tmp.index.values, tmp[utility_col].values, linewidth=1.0, label="utility")
                        ax.legend()
                    ax.set_title(f"{bundle['run_name']} - Utility by Document")
                    ax.set_xlabel("Document index")
                    ax.set_ylabel(utility_col)
                    plt.tight_layout()
                    fig.savefig(run_dir / "utility_by_document.png", dpi=150, bbox_inches="tight")
                    plt.close(fig)

    return out_dir


if manifest.empty:
    print("Nothing to export yet because no runs were discovered.")
else:
    export_dir = export_dashboard_outputs(manifest_profiled, cross_summary, run_bundles, EXPORT_ROOT)
    print("Exported dashboard artifacts to:")
    print(export_dir)